# Day 06：函数（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 1 周**：第 1 周：Python 核心语法与数据结构基石  
> **GitLab 工单**：`STARMIND-006` ｜ **开发分支**：`feat/day06-functions-refactor`  
> **配套 HTML5 课件**：`courseware/day06/index.html` ｜ **独立版本快照**：`versions/day06/`

---

## 🎙️ 老陈开场旁白

前五天咱们写了不少实用片段，但你有没有发现：代码里重复逻辑开始变多了，参数一多就容易传错顺序？今天 STARMIND-006 咱们把「函数」彻底吃透：位置参数、关键字参数、默认参数、`*args`、`**kwargs`、`lambda` 匿名函数以及递归。大厂写 AI 工具箱最讲究「函数式管道（Pipeline）」——把清洗、截断、脱敏做成可插拔的小函数，像乐高一样自由组合！另外顺手用递归解决林产品提的一个真实需求：遍历星辰知识库多层嵌套文件夹 JSON 树，统计所有文档数量与路径。

### 承上启下
- **昨日基线**：昨天咱们攻克了嵌套字典和 JSON 序列化，已经能安全解析模拟大模型 API 返回的响应并落盘。
- **明日预告**：明天（Day 7）是第一周复习与周测日！咱们会把 Day 1~6 所有核心知识串成一张网，完成周测并独立交付「星辰企业通讯录管理系统 + JSON 持久化」。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day06'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-006）

星辰智脑的文本预处理步骤越来越多（去噪、脱敏、截断、加前缀），硬编码串行调用难以按业务线灵活编排；同时知识库目录支持无限层级嵌套，需要递归统计文档清单。

**今日迭代目标：**
- 掌握位置/关键字/默认参数以及 `*args`、`**kwargs` 的透传封装模式。
- 熟练使用 `lambda` 配合 `sorted` / `filter` / `map` 完成数据排序与管道组合。
- 理解递归的终止条件与分解思路，安全遍历嵌套树形结构。

**验收标准：**
- 实现 `run_pipeline(text, *processors)`，支持按顺序动态组合任意数量的文本处理函数。
- 实现 `build_chat_request(model, *messages, **extra_params)`，灵活组装 API 请求体。
- 实现递归函数 `flatten_kb_tree(node)`，提取任意深度知识库树中的所有文档路径。

---

## 📚 课堂核心技术笔记

### 1. 1. 参数顺序与 *args / **kwargs 解包

函数参数严格遵循顺序：`位置参数 -> 默认参数 -> *args -> 仅关键字参数 -> **kwargs`。`*args` 把多余位置参数打包成 `tuple`，`**kwargs` 把多余关键字参数打包成 `dict`——在封装大模型 SDK 客户端时，用 `**kwargs` 透传可选超参（`temperature`, `top_p`, `seed`）是最标准写法。

> **⚠️ 老陈避坑**：致命经典坑：`def add_msg(msg, history=[]):` 中默认列表在函数定义时只创建一次，多次调用会互相污染！正确写法：`def add_msg(msg, history=None): history = [] if history is None else history`。

### 2. 2. lambda 匿名函数与高阶排序

`lambda 参数: 表达式` 适合写简短的一次性逻辑，最常用于 `sorted(docs, key=lambda d: d['score'], reverse=True)`。如果逻辑超过一行或需要复用，老陈强烈建议用 `def` 定义具名函数并写上 docstring。

> **⚠️ 老陈避坑**：在循环里创建 lambda 容易踩「闭包延迟绑定」的坑：`[lambda: i for i in range(3)]` 三个函数调用结果全是 2，需要写成 `lambda i=i: i` 绑定当前值。

### 3. 3. 递归两要素：基准条件与缩小问题规模

写递归前先写 `if 到达叶子/边界: return ...`（Base Case），再调用自身处理更小规模的子问题。Python 默认最大递归深度为 1000（`sys.getrecursionlimit()`），适合目录树、JSON 树、语法树遍历，不适合百万级线性链表。

> **⚠️ 老陈避坑**：漏写基准条件或递归参数没有向边界收敛，会触发 `RecursionError: maximum recursion depth exceeded`。


## 💻 实操步骤 1：步骤一：构建函数式文本管道与动态请求组装器

- **目标文件**：`versions/day06/src/day06_text_pipeline.py`
- **老陈代码旁白**：咱们把前几天零散的文本清洗步骤重构为高内聚的函数管道，并用 `*args` / `**kwargs` 写出灵活的 Chat API 请求构造器。


In [ ]:
"""
STARMIND-006 模块一：函数式文本处理管道与动态 API 请求构造器。
演示默认参数避坑（None 哨兵）、*args 管道组合、**kwargs 参数透传与 lambda 处理器。
"""

from typing import Callable, Dict, Any, List, Optional


def strip_and_collapse(text: str) -> str:
    """处理器 1：去除首尾空白并折叠连续空格与换行。"""
    return " ".join(text.replace("\u3000", " ").split())


def redact_secret_key(text: str) -> str:
    """处理器 2：对意外粘贴进提问中的 sk- 密钥片段做简单遮蔽。"""
    tokens = text.split(" ")
    safe_tokens = ["sk-***REDACTED***" if t.startswith("sk-") else t for t in tokens]
    return " ".join(safe_tokens)


def run_pipeline(text: str, *processors: Callable[[str], str]) -> str:
    """
    函数式管道调度器：将输入文本依次送入 *processors 中的每个函数处理。
    支持传入具名函数或 lambda 匿名函数。
    """
    result = text
    for fn in processors:
        result = fn(result)
    return result


def build_chat_request(
    model: str,
    *user_messages: str,
    system_prompt: Optional[str] = None,
    **extra_params: Any,
) -> Dict[str, Any]:
    """
    动态构建大模型 Chat Completions 请求体。
    - *user_messages 接收一条或多条用户消息
    - **extra_params 透传 temperature、max_tokens 等可选超参
    """
    # 安全初始化可变列表（绝不用 messages=[] 作为默认参数）
    messages: List[Dict[str, str]] = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    for msg in user_messages:
        messages.append({"role": "user", "content": msg})

    payload: Dict[str, Any] = {
        "model": model,
        "messages": messages,
    }
    # 过滤掉值为 None 的额外超参后合并
    for key, value in extra_params.items():
        if value is not None:
            payload[key] = value
    return payload


if __name__ == "__main__":
    raw = "  帮我查下  sk-secret123456  为什么报 401 错误？ \n "
    # 自由组合两个具名处理器 + 一个 lambda 截断处理器
    cleaned = run_pipeline(
        raw,
        strip_and_collapse,
        redact_secret_key,
        lambda s: s[:40],
    )
    req = build_chat_request(
        "deepseek-chat",
        cleaned,
        system_prompt="你是星辰智脑运维助手",
        temperature=0.2,
        max_tokens=512,
    )
    print("清洗结果:", cleaned)
    print("请求体  :", req)


### 🔍 步骤 1 代码深度解析

1. `run_pipeline` 的 `*processors` 允许调用方随心所欲增减步骤，甚至临时传一个 `lambda s: s[:40]` 做截断，完全符合开闭原则。
2. `build_chat_request` 展示了一个非常标准的 SDK 函数签名设计：必填 `model` 放前面，变长 `*user_messages` 居中，仅关键字参数 `system_prompt` 与 `**extra_params` 殿后。

**预期运行输出参考：**
```text
清洗结果: 帮我查下 sk-***REDACTED*** 为什么报 401 错误？
请求体  : {'model': 'deepseek-chat', 'messages': [{'role': 'system', 'content': '你是星辰智脑运维助手'}, {'role': 'user', 'content': '帮我查下 sk-***REDACTED*** 为什么报 401 错误？'}], 'temperature': 0.2, 'max_tokens': 512}
```


## 💻 实操步骤 2：步骤二：使用递归与 lambda 排序遍历知识库嵌套目录树

- **目标文件**：`versions/day06/src/day06_kb_tree.py`
- **老陈代码旁白**：现在用递归解决星辰知识库多层目录统计问题，并用 `lambda` 按文档大小降序找出体积最大的 Top-N 文档。


In [ ]:
"""
STARMIND-006 模块二：知识库嵌套目录树递归解析与 Top-N 排序器。
演示递归函数（基准条件 + 递归分解）、lambda 排序与深度安全控制。
"""

from typing import Dict, Any, List


def flatten_kb_tree(
    node: Dict[str, Any],
    prefix: str = "",
    max_depth: int = 20,
) -> List[Dict[str, Any]]:
    """
    递归遍历知识库嵌套目录树，提取所有文件节点及其完整逻辑路径。
    每个节点格式：
      - 文件: {"type": "file", "name": "faq.md", "size_kb": 12}
      - 目录: {"type": "dir", "name": "客服部", "children": [...]}
    """
    if max_depth < 0:
        raise RecursionError("知识库目录嵌套层级超过安全上限")

    name = str(node.get("name", ""))
    current_path = f"{prefix}/{name}" if prefix else name

    # 基准条件（Base Case）：到达文件叶节点，直接返回单元素列表
    if node.get("type") == "file":
        return [{"path": current_path, "size_kb": int(node.get("size_kb", 0))}]

    # 递归步骤：遍历目录的所有子节点并汇总结果
    results: List[Dict[str, Any]] = []
    for child in node.get("children", []):
        results.extend(flatten_kb_tree(child, prefix=current_path, max_depth=max_depth - 1))
    return results


def top_largest_docs(tree: Dict[str, Any], top_n: int = 3) -> List[Dict[str, Any]]:
    """找出知识库中体积最大的前 top_n 份文档（使用 lambda 多键排序）。"""
    all_files = flatten_kb_tree(tree)
    # 先按 size_kb 降序，体积相同时按 path 字母升序，保证排序稳定
    ranked = sorted(all_files, key=lambda doc: (-doc["size_kb"], doc["path"]))
    return ranked[:top_n]


if __name__ == "__main__":
    sample_kb = {
        "type": "dir",
        "name": "StarMind_KB",
        "children": [
            {"type": "file", "name": "readme.md", "size_kb": 5},
            {
                "type": "dir",
                "name": "研发部",
                "children": [
                    {"type": "file", "name": "api_spec.md", "size_kb": 48},
                    {
                        "type": "dir",
                        "name": "新人培训",
                        "children": [{"type": "file", "name": "python_guide.md", "size_kb": 120}],
                    },
                ],
            },
            {"type": "file", "name": "refund_policy.md", "size_kb": 32},
        ],
    }
    print("Top 2 最大文档:", top_largest_docs(sample_kb, top_n=2))


### 🔍 步骤 2 代码深度解析

1. `flatten_kb_tree` 加入了 `max_depth` 防御参数，防止恶意构造的畸形 JSON 引发栈溢出。
2. `key=lambda doc: (-doc['size_kb'], doc['path'])` 是 Python 经典多条件混合排序技巧：数值取负实现降序，字符串保持升序。

**预期运行输出参考：**
```text
Top 2 最大文档: [{'path': 'StarMind_KB/研发部/新人培训/python_guide.md', 'size_kb': 120}, {'path': 'StarMind_KB/研发部/api_spec.md', 'size_kb': 48}]
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day06/tests/test_day06.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-006 自动化单元测试。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day06_text_pipeline import strip_and_collapse, redact_secret_key, run_pipeline, build_chat_request
from day06_kb_tree import flatten_kb_tree, top_largest_docs


def test_text_pipeline_and_request_builder():
    out = run_pipeline("  key= sk-abc123  hello  ", strip_and_collapse, redact_secret_key)
    assert out == "key= sk-***REDACTED*** hello"

    req1 = build_chat_request("qwen-plus", "你好", temperature=0.5, seed=None)
    req2 = build_chat_request("qwen-plus", "第二条")
    assert len(req1["messages"]) == 1
    assert len(req2["messages"]) == 1  # 确保两次调用消息列表完全隔离
    assert "seed" not in req1
    assert req1["temperature"] == 0.5


def test_kb_tree_recursion_and_ranking():
    tree = {
        "type": "dir",
        "name": "root",
        "children": [
            {"type": "file", "name": "a.md", "size_kb": 10},
            {
                "type": "dir",
                "name": "sub",
                "children": [
                    {"type": "file", "name": "b.md", "size_kb": 50},
                    {"type": "file", "name": "c.md", "size_kb": 50},
                ],
            },
        ],
    }
    flat = flatten_kb_tree(tree)
    assert len(flat) == 3
    top2 = top_largest_docs(tree, top_n=2)
    assert [d["path"] for d in top2] == ["root/sub/b.md", "root/sub/c.md"]

    with pytest.raises(RecursionError):
        flatten_kb_tree(tree, max_depth=0)

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：修复可变默认参数 Bug

**题目要求：**
下面有 Bug 的函数 `def append_message(msg, history=[]): history.append(msg); return history` 会导致不同用户对话串台。请重写安全版本 `append_message(msg: str, history: list | None = None) -> list[str]`，并验证两次独立调用互不干扰。

**思路提示：** 用 `None` 作为默认值，且为了不意外修改外部传入列表，建议在函数内拷贝或明确约定行为。


In [ ]:
"""Day 6 课后题 1：修复可变默认参数导致的对话串台 Bug。"""

from typing import List, Optional


def append_message(msg: str, history: Optional[List[str]] = None) -> List[str]:
    """安全追加对话消息，每次未传 history 时创建全新列表。"""
    records = [] if history is None else list(history)
    records.append(msg)
    return records


if __name__ == "__main__":
    user_a = append_message("你好，我是A")
    user_b = append_message("你好，我是B")
    print("用户A:", user_a)
    print("用户B:", user_b)


**答案解析与评分标准：**

同时使用了 `None` 哨兵和 `list(history)` 防御性拷贝，彻底消除状态串台隐患。


### 作业 2（企业实战）：企业实战：带关键词过滤的文档检索排序器

**题目要求：**
编写 `search_docs(docs: list[dict], keyword: str, min_score: float = 0.5) -> list[dict]`，用列表推导式 + `lambda` 筛选出标题包含关键词且得分 `>= min_score` 的文档，按得分降序返回。

**思路提示：** 先过滤再 `sorted(..., key=lambda d: d['score'], reverse=True)`。


In [ ]:
"""Day 6 课后题 2：关键词过滤与相关度排序检索器。"""

from typing import List, Dict, Any


def search_docs(
    docs: List[Dict[str, Any]], keyword: str, min_score: float = 0.5
) -> List[Dict[str, Any]]:
    """按关键词和最低分阈值过滤文档，并按相关度降序排列。"""
    kw = keyword.strip().lower()
    matched = [
        d for d in docs
        if kw in str(d.get("title", "")).lower() and float(d.get("score", 0.0)) >= min_score
    ]
    return sorted(matched, key=lambda d: d["score"], reverse=True)


if __name__ == "__main__":
    kb = [
        {"title": "Python 异步编程指南", "score": 0.92},
        {"title": "Python 安装手册", "score": 0.41},
        {"title": "FastAPI + Python 实战", "score": 0.85},
    ]
    print(search_docs(kb, "python", min_score=0.5))


**答案解析与评分标准：**

大小写不敏感匹配 + 阈值过滤 + lambda 降序，是简化版 RAG 重排（Rerank）的核心逻辑。


### 作业 3（架构进阶）：架构进阶：递归统计嵌套 JSON 中所有数值型 Token 总和

**题目要求：**
一个复杂的多 Agent 调用日志 JSON 中，各层子任务字典里都可能包含名为 `'tokens'` 的整数键。编写递归函数 `sum_nested_tokens(data: Any) -> int`，无论嵌套多少层 dict/list，准确汇总所有 `'tokens'` 之和。

**思路提示：** 若 `isinstance(data, dict)` 则累加 `data['tokens']`（若为 int 且非 bool）并递归其所有 values；若 `isinstance(data, list)` 则递归其每个元素。


In [ ]:
"""Day 6 课后题 3：递归汇总嵌套多 Agent 日志中的总 Token 消耗。"""

from typing import Any


def sum_nested_tokens(data: Any) -> int:
    """递归遍历任意深度 dict/list，汇总所有键名为 'tokens' 的整数值。"""
    total = 0
    if isinstance(data, dict):
        for k, v in data.items():
            # 注意：Python 中 bool 是 int 的子类，需排除 True/False
            if k == "tokens" and isinstance(v, int) and not isinstance(v, bool):
                total += v
            else:
                total += sum_nested_tokens(v)
    elif isinstance(data, list):
        for item in data:
            total += sum_nested_tokens(item)
    return total


if __name__ == "__main__":
    trace = {
        "agent": "planner",
        "tokens": 150,
        "subtasks": [
            {"agent": "search", "tokens": 300},
            {"agent": "writer", "tokens": 450, "retries": [{"tokens": 100}]},
        ],
    }
    print("多 Agent 总消耗 Token:", sum_nested_tokens(trace))


**答案解析与评分标准：**

特别注意 `not isinstance(v, bool)` 这个高阶细节——因为 `isinstance(True, int)` 为 `True`，漏掉判断会把布尔值当 1 算进账单里！


---

## 🎯 老陈收尾复盘

写完今天的函数式管道，回头再看 Day 2 的代码是不是觉得清爽了一大截？牢记今天最大的坑：**永远不要用可变对象（如 `[]` 或 `{}`）作为函数默认参数**！记住用 `None` 占位再在函数体内初始化。明天周测实战，今晚好好复盘一下前六天的知识点。
